<div align="center">

<h2>Examen de Recuperación de Información</h2>
<h2>ESCUELA POLITÉCNICA NACIONAL</h2>

<h3>FACULTAD DE INGENIERÍA DE SISTEMAS</h3>


<p><strong>Estudiante:</strong> Estéfano Condoy</p>

<p><strong>Fecha:</strong> 18/11/2026</p>

</div>

<hr>




## 1. Objetivo e Importaciones

En este trabajo armé un buscador sobre SciFact. Primero preparé e indexé los
documentos. Luego usé TF-IDF con similitud coseno y BM25 para ordenar los
resultados. Al final comparé los rankings con las referencias de la partición
de prueba y revisé la respuesta del servicio web.

**Archivo de datos.** Ejecuté el cuaderno con `scifact.zip` de mi carpeta
Descargas. Dejé guardadas las salidas de todas las celdas para que se puedan
revisar al abrir el archivo.

Aquí construyo un buscador con el corpus que me asignaron. Primero preparo los documentos, luego creo el índice invertido, comparo dos modelos y finalmente los uso en una API.

Mi corpus asignado fue : **SciFact**

In [4]:
import csv
import io
import json
import re
import unicodedata
from collections import Counter, defaultdict
from math import log, log1p, log2, sqrt
from pathlib import Path
from statistics import mean, median
from zipfile import ZipFile

## 2. Corpus asignado

Cargué el ZIP de SciFact en formato BEIR. `corpus.jsonl` trae títulos y textos
de artículos. `queries.jsonl` contiene las afirmaciones en inglés que sirven
de consultas. `qrels/test.tsv` me indica qué documentos cuentan como relevantes
para cada consulta de prueba. Por eso cargo todas las consultas disponibles
y tomo los IDs de los qrels de `test` para decidir cuáles entran en la evaluación.

In [11]:
def EC_cargar(ruta_zip):
    with ZipFile(ruta_zip) as archivo:
        archivos = set(archivo.namelist())
        corpus = next(
            (nombre for nombre in sorted(archivos)
             if nombre.rsplit("/", 1)[-1] == "corpus.jsonl"),
            None,
        )
        if corpus is None:
            raise ValueError("El ZIP debe contener corpus.jsonl.")
        carpeta = corpus.removesuffix("corpus.jsonl")

        def leer_jsonl(nombre):
            with archivo.open(carpeta + nombre) as origen:
                return {
                    registro["_id"]: registro
                    for linea in origen
                    if linea.strip()
                    for registro in [json.loads(linea)]
                }

        documentos = leer_jsonl("corpus.jsonl")
        consultas = {
            identificador: registro["text"]
            for identificador, registro in leer_jsonl("queries.jsonl").items()
        }
        qrels = defaultdict(dict)
        with archivo.open(carpeta + "qrels/test.tsv") as origen:
            filas = csv.DictReader(io.TextIOWrapper(origen, encoding="utf-8"), delimiter="\t")
            for fila in filas:
                qrels[fila["query-id"]][fila["corpus-id"]] = int(fila["score"])

    assert set(qrels) <= consultas.keys()
    assert all(juicios.keys() <= documentos.keys() for juicios in qrels.values())
    assert (len(documentos), len(consultas), len(qrels)) == (5183, 1109, 300), (
        "El archivo esperado es el ZIP de SciFact en formato BEIR."
    )
    return documentos, consultas, dict(qrels)

EC_RUTA_ZIP = Path(r"C:\Users\Estefano Condoy\Downloads\scifact.zip")
EC_documentos, EC_consultas, EC_qrels = EC_cargar(EC_RUTA_ZIP)
EC_juicios = [grado for juicios in EC_qrels.values() for grado in juicios.values()]
print(f"ZIP utilizado: {EC_RUTA_ZIP}")
print(f"Documentos: {len(EC_documentos):,}")
print(f"Consultas disponibles: {len(EC_consultas):,}")
print(f"Consultas con qrels de test: {len(EC_qrels):,}")
print(f"Pares consulta-documento en test: {len(EC_juicios):,}")
print(f"Grados de relevancia en test: {dict(Counter(EC_juicios))}")
EC_id_ejemplo = min(EC_qrels, key=int)
print(f"Ejemplo de consulta de test [{EC_id_ejemplo}]: {EC_consultas[EC_id_ejemplo]}")
print("Documentos de referencia:", EC_qrels[EC_id_ejemplo])

ZIP utilizado: C:\Users\Estefano Condoy\Downloads\scifact.zip
Documentos: 5,183
Consultas disponibles: 1,109
Consultas con qrels de test: 300
Pares consulta-documento en test: 339
Grados de relevancia en test: {1: 339}
Ejemplo de consulta de test [1]: 0-dimensional biomaterials show inductive properties.
Documentos de referencia: {'31715818': 1}


Encontré **5.183 documentos**, **1.109 consultas** en el archivo general y
**300 consultas** para evaluar. Los **339 pares** anotados en `test` tienen
grado de relevancia 1. Una consulta puede tener varios documentos relevantes.
Al calcular las métricas, los documentos sin un juicio positivo para esa
consulta aportan cero.

## 3. Requisitos del sistema
### 3.1. Preparación e indexación

**Preprocesamiento.** Usé la misma función para los documentos y las consultas.
Paso todo a minúsculas, normalizo las letras con acento y extraigo secuencias
de letras de la A a la Z o dígitos. Así puedo conservar números y términos
como `p53`. Quito palabras frecuentes en inglés, como `the` y `and`, porque
aparecen en muchos documentos. Mantengo `no`, `not` y `nor`, ya que la
negación cambia el sentido de las afirmaciones.

Apliqué *stemming* con reglas sencillas para plurales y terminaciones. Por
ejemplo, `studies` y `study` quedan con la misma raíz. Algunas palabras
quedan recortadas, como `activated` y `activating`, que se convierten en
`activat`. Esas raíces se usan para comparar palabras durante la búsqueda.

In [16]:
EC_STOPWORDS = frozenset("""
a about above after again all also am an and any are as at be because been before being
below between both but by can could did do does doing down during each few for from further
had has have having he her here hers herself him himself his how i if in into is it its
itself just may me might more most must my myself of off on once only or other our ours
ourselves out over own same she should so some such than that the their theirs them
themselves then there these they this those through to too under until up very was we
were what when where which while who whom why will with would you your yours yourself
yourselves s t d m re ve ll
""".split())
EC_PATRON = re.compile(r"[a-z0-9]+")


def EC_raiz(termino):
    # Stemming ligero. Conservo números, términos alfanuméricos y palabras cortas.
    if len(termino) < 4 or not termino.isalpha():
        return termino
    if len(termino) >= 5 and termino.endswith("ies"):
        return termino[:-3] + "y"
    if len(termino) >= 5 and termino.endswith(("sses", "shes", "ches", "xes", "zzes")):
        return termino[:-2]
    if termino.endswith("s") and not termino.endswith(("ss", "us", "is")):
        termino = termino[:-1]

    # Relaciono variantes como activate, activated y activating.
    for sufijo in ("ing", "ed"):
        if termino.endswith(sufijo):
            base = termino[:-len(sufijo)]
            if len(base) >= 3 and re.search(r"[aeiouy]", base):
                termino = base
                if termino.endswith(("bb", "dd", "ff", "gg", "mm", "nn", "pp", "rr", "tt", "zz")):
                    termino = termino[:-1]
            break
    if len(termino) >= 5 and termino.endswith("e") and not termino.endswith("ee"):
        termino = termino[:-1]
    return termino


def EC_tokenizar(texto):
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(letra for letra in texto if not unicodedata.category(letra).startswith("M"))
    texto = re.sub(r"n['’]t(?![a-z0-9_])", " not", texto)
    palabras = EC_PATRON.findall(texto)
    return [EC_raiz(palabra) for palabra in palabras if palabra not in EC_STOPWORDS]

EC_texto_ejemplo = "The studies are activating p53, but p53 is not activated."
print(EC_texto_ejemplo)
print(EC_tokenizar(EC_texto_ejemplo))
print(f"Consulta {EC_id_ejemplo}:", EC_tokenizar(EC_consultas[EC_id_ejemplo]))

The studies are activating p53, but p53 is not activated.
['study', 'activat', 'p53', 'p53', 'not', 'activat']
Consulta 1: ['0', 'dimensional', 'biomaterial', 'show', 'inductiv', 'property']


En el ejemplo, `the` y `are` se eliminan. `p53` se conserva, las dos
variantes de *activate* comparten raíz y `not` sigue en la lista. El buscador
compara coincidencias de términos y esa coincidencia por sí sola no permite
decidir si un artículo apoya o contradice lo que dice la consulta.

**Índice invertido.** Para cada término guardé qué documentos lo contienen
y cuántas veces aparece en cada uno. Indexé el título y el resumen juntos como
un solo documento y conservé el identificador original. Las listas del índice
usan un número de fila para ubicar los resultados. Antes de responder, esa
fila se convierte otra vez al ID de SciFact. También guardé la longitud de
cada documento y los pesos que necesitan ambos métodos.

In [19]:
class EC_Indice:
    def __init__(self, documentos, k1=1.2, b=0.75):
        self.ids = sorted(documentos)
        self.documentos = documentos
        self.k1 = k1
        self.b = b
        self.longitudes = []
        self.postings = defaultdict(list)
        for fila, identificador in enumerate(self.ids):
            doc = documentos[identificador]
            texto = f"{doc.get('title', '')} {doc.get('text', '')}"
            tokens = EC_tokenizar(texto)
            self.longitudes.append(len(tokens))
            for termino, tf in Counter(tokens).items():
                self.postings[termino].append((fila, tf))

        self.vocabulario = sorted(self.postings)
        self.longitud_media = mean(self.longitudes)
        total = len(self.ids)
        self.idf_tfidf = {}
        self.idf_bm25 = {}
        norma_cuadrada = [0.0] * total
        for termino in self.vocabulario:
            apariciones = self.postings[termino]
            df = len(apariciones)
            idf = 1 + log((1 + total) / (1 + df))
            self.idf_tfidf[termino] = idf
            self.idf_bm25[termino] = log1p((total - df + 0.5) / (df + 0.5))
            for fila, tf in apariciones:
                peso = tf * idf
                norma_cuadrada[fila] += peso * peso
        self.normas = [sqrt(valor) for valor in norma_cuadrada]

EC_indice = EC_Indice(EC_documentos)
EC_termino_ejemplo = "biomaterial"
EC_apariciones = EC_indice.postings[EC_termino_ejemplo]
print(f"Términos del vocabulario: {len(EC_indice.vocabulario):,}")
print(f"Longitud media y mediana: {EC_indice.longitud_media:.2f} y "
      f"{median(EC_indice.longitudes):g} tokens")
print(f"Documentos que contienen «{EC_termino_ejemplo}»: {len(EC_apariciones)}")
print("Primeras entradas (ID de SciFact, frecuencia):",
      [(EC_indice.ids[fila], tf) for fila, tf in EC_apariciones[:5]])

Términos del vocabulario: 29,382
Longitud media y mediana: 151.22 y 140 tokens
Documentos que contienen «biomaterial»: 1
Primeras entradas (ID de SciFact, frecuencia): [('13231899', 1)]


El vocabulario quedó en **29.382 términos** y cada documento tiene, en promedio,
**151,22 tokens** después de la limpieza. La consulta puede empezar por las
listas de sus términos, lo que evita recorrer todos los resúmenes para cada
búsqueda. Los IDs que aparecen en el ejemplo están en el formato de los qrels.

### 3.2. Modelos de recuperación

- **TF-IDF con coseno.** Multiplico la frecuencia de cada término por su IDF.
  Uso IDF suavizada, calculada como `1 + ln((1 + N)/(1 + df))`, donde `N` es
  el número de documentos y `df` es cuántos contienen el término. Calculo el
  producto punto entre los pesos de la consulta y el documento, y lo divido
  por el producto de sus normas. Así puedo comparar textos de distintos tamaños.
- **BM25.** Uso `k1 = 1.2` y `b = 0.75`. La primera constante hace que repetir
  una palabra aporte cada vez menos. La segunda ajusta ese aporte según la
  longitud del documento respecto a la media. Apliqué los mismos parámetros a
  todas las consultas, sin ajustarlos con los qrels de prueba.

Ambos rankings ordenan sus propias puntuaciones de mayor a menor. Si hay un
empate, uso el orden de los IDs originales. Los términos que el índice
desconoce aportan cero. Si una consulta no tiene coincidencias, devuelve
una lista vacía.

In [26]:
class EC_Buscador(EC_Indice):
    def puntuar(self, consulta, modelo, top_k):
        if modelo not in {"tfidf", "bm25"}:
            raise ValueError("El modelo debe ser bm25 o tfidf.")
        frecuencias = Counter(EC_tokenizar(consulta))
        terminos = sorted(frecuencias.keys() & self.postings.keys())
        puntos = defaultdict(float)

        if modelo == "tfidf":
            pesos = {
                termino: frecuencias[termino] * self.idf_tfidf[termino]
                for termino in terminos
            }
            norma_consulta = sqrt(sum(peso * peso for peso in pesos.values()))
            for termino, peso_consulta in pesos.items():
                idf = self.idf_tfidf[termino]
                for fila, tf in self.postings[termino]:
                    puntos[fila] += peso_consulta * (tf * idf)
            for fila in puntos:
                puntos[fila] /= norma_consulta * self.normas[fila]
        else:
            # Cada término distinto de la consulta aporta una vez en BM25.
            for termino in terminos:
                idf = self.idf_bm25[termino]
                for fila, tf in self.postings[termino]:
                    longitud = self.longitudes[fila]
                    ajuste = self.k1 * (1 - self.b + self.b * longitud / self.longitud_media)
                    puntos[fila] += idf * (tf * (self.k1 + 1) / (tf + ajuste))

        # La fila sigue el orden lexicográfico de los IDs originales.
        return sorted(puntos.items(), key=lambda par: (-par[1], par[0]))[:top_k]

    def buscar(self, consulta, modelo="bm25", top_k=10):
        resultados = []
        for posicion, (fila, score) in enumerate(self.puntuar(consulta, modelo, top_k), 1):
            identificador = self.ids[fila]
            doc = self.documentos[identificador]
            resultados.append({
                "rank": posicion,
                "document_id": identificador,
                "score": score,
                "title": doc.get("title", ""),
                "snippet": EC_fragmento(doc.get("text", "")),
            })
        return {"model": modelo, "results": resultados}


def EC_fragmento(texto, limite=250):
    texto = " ".join(texto.split())
    if len(texto) <= limite:
        return texto
    return texto[:limite].rsplit(" ", 1)[0] + "…"

EC_buscador = EC_Buscador(EC_documentos)
EC_demo_id = "3"
print(EC_consultas[EC_demo_id])
print("ID de referencia:", ", ".join(EC_qrels[EC_demo_id]))
for EC_modelo in ("tfidf", "bm25"):
    print(f"\n{EC_modelo.upper()} · 5 primeros")
    for resultado in EC_buscador.buscar(EC_consultas[EC_demo_id], EC_modelo, 5)["results"]:
        print(f"{resultado['rank']}. {resultado['document_id']:>8} | "
              f"{resultado['score']:.6f} | {resultado['title']}")

1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.
ID de referencia: 14717500

TFIDF · 5 primeros
1. 23389795 | 0.397261 | Common and rare variants in multifactorial susceptibility to common diseases
2.  2739854 | 0.343553 | Rare and common variants: twenty arguments
3. 14717500 | 0.275444 | Rare Variants Create Synthetic Genome-Wide Associations
4.  4632921 | 0.243011 | Large-Scale Profiling Reveals the Influence of Genetic Variation on Gene Expression in Human Induced Pluripotent Stem Cells.
5.  4378885 | 0.211410 | Understanding mechanisms underlying human gene expression variation with RNA sequencing

BM25 · 5 primeros
1. 14717500 | 30.102562 | Rare Variants Create Synthetic Genome-Wide Associations
2.  4414547 | 28.506795 | Mosaic PPM1D mutations are associated with predisposition to breast and ovarian cancer
3.  2739854 | 28.397966 | Rare and common variants: twenty arguments
4.  437

En la consulta 3, el documento de referencia **14717500** aparece en ambos
rankings. TF-IDF lo pone tercero y BM25 lo pone primero. El valor numérico
del puntaje tiene una escala distinta en cada modelo. Para comparar cuál
recupera mejor, revisé las posiciones y las métricas con los qrels.

### 3.3. Servicio web

`GET /health` devuelve el estado, el corpus, la cantidad de documentos y los
modelos disponibles. `POST /search` recibe `query`, `model` y `top_k` en
JSON. Cada resultado tiene `rank`, `document_id`, `score`, `title` y
`snippet`. El fragmento sirve para dar una vista previa del resumen.

La versión local usa el buscador de este cuaderno. El despliegue de la API
utiliza los postings, identificadores y pesos exportados desde ese mismo
índice y aplica las mismas reglas al recibir consultas. En esta celda
reviso primero el formato de la respuesta con un ejemplo de SciFact.

In [28]:
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.error import HTTPError
from urllib.request import Request, urlopen


def EC_validar(solicitud, limite):
    if not isinstance(solicitud, dict):
        raise ValueError("Envía un objeto JSON.")
    consulta = solicitud.get("query")
    modelo = solicitud.get("model", "bm25")
    top_k = solicitud.get("top_k", 10)
    if not isinstance(consulta, str) or not consulta.strip():
        raise ValueError("query debe ser texto y contener al menos un carácter.")
    if len(consulta) > 4000:
        raise ValueError("query admite hasta 4000 caracteres.")
    if modelo not in ("bm25", "tfidf"):
        raise ValueError("model debe ser bm25 o tfidf.")
    if type(top_k) not in (int, float) or not 1 <= top_k <= limite or top_k != int(top_k):
        raise ValueError(f"top_k debe ser un entero entre 1 y {limite}.")
    return consulta, modelo, int(top_k)


def EC_responder(buscador, metodo, ruta, entrada=None):
    if ruta == "/health" and metodo == "GET":
        return 200, {
            "status": "ok",
            "corpus": "SciFact",
            "documents": len(buscador.ids),
            "models": ["bm25", "tfidf"],
        }
    if ruta == "/search" and metodo == "POST":
        try:
            argumentos = EC_validar(entrada, len(buscador.ids))
        except ValueError as error:
            return 400, {"error": str(error)}
        return 200, buscador.buscar(*argumentos)
    if ruta in ("/health", "/search"):
        return 405, {"error": "Método no permitido."}
    return 404, {"error": "Ruta no encontrada."}


def EC_servidor_local(buscador, puerto=0):
    class EC_Manejador(BaseHTTPRequestHandler):
        def atender(self):
            ruta = self.path.partition("?")[0]
            entrada = None
            if self.command == "POST" and ruta == "/search":
                try:
                    longitud = int(self.headers.get("Content-Length", "0"))
                    if not 0 < longitud <= 65536:
                        raise ValueError("Longitud inválida.")
                    entrada = json.loads(self.rfile.read(longitud))
                except (ValueError, UnicodeError):
                    self.enviar(400, {"error": "El cuerpo debe ser JSON válido."})
                    return
            codigo, respuesta = EC_responder(buscador, self.command, ruta, entrada)
            self.enviar(codigo, respuesta)

        def enviar(self, codigo, respuesta):
            carga = json.dumps(respuesta, ensure_ascii=False).encode("utf-8")
            self.send_response(codigo)
            self.send_header("Content-Type", "application/json; charset=utf-8")
            self.send_header("Content-Length", str(len(carga)))
            self.end_headers()
            self.wfile.write(carga)

        do_GET = atender
        do_POST = atender

        def log_message(self, formato, *argumentos):
            pass

    return ThreadingHTTPServer(("127.0.0.1", puerto), EC_Manejador)


def EC_peticion(base, ruta, solicitud=None):
    datos = None if solicitud is None else json.dumps(solicitud).encode("utf-8")
    peticion = Request(base + ruta, data=datos, headers={"Content-Type": "application/json"})
    try:
        with urlopen(peticion, timeout=15) as respuesta:
            return respuesta.status, json.load(respuesta)
    except HTTPError as error:
        with error:
            return error.code, json.load(error)

EC_solicitud = {"query": EC_consultas[EC_demo_id], "model": "bm25", "top_k": 2}
EC_codigo, EC_datos = EC_responder(EC_buscador, "POST", "/search", EC_solicitud)
print(f"HTTP {EC_codigo}")
print(json.dumps(EC_datos, indent=2, ensure_ascii=False))

HTTP 200
{
  "model": "bm25",
  "results": [
    {
      "rank": 1,
      "document_id": "14717500",
      "score": 30.10256234771158,
      "title": "Rare Variants Create Synthetic Genome-Wide Associations",
      "snippet": "Genome-wide association studies (GWAS) have now identified at least 2,000 common variants that appear associated with common diseases or related traits (http://www.genome.gov/gwastudies), hundreds of which have been convincingly replicated. It is…"
    },
    {
      "rank": 2,
      "document_id": "4414547",
      "score": 28.50679471303898,
      "title": "Mosaic PPM1D mutations are associated with predisposition to breast and ovarian cancer",
      "snippet": "Improved sequencing technologies offer unprecedented opportunities for investigating the role of rare genetic variation in common disease. However, there are considerable challenges with respect to study design, data analysis and replication. Using…"
    }
  ]
}


En el ejemplo salen dos resultados ordenados. El ID de cada resultado
corresponde al documento completo y el fragmento se arma a partir del resumen.

## 4. Evaluación de la recuperación

Usé las **300 consultas de test** identificadas por las filas de
`qrels/test.tsv`. El código imprime todos esos IDs y muestra el texto
de las primeras consultas como referencia. Para cada consulta busqué los
primeros 10 documentos de cada modelo y comparé los IDs con sus qrels.

- **Precision@10** mide cuántos de los 10 puestos contienen documentos relevantes.
  Si el buscador devuelve menos de 10 resultados, mantengo 10 como denominador.
- **Recall@10** divide los relevantes encontrados entre todos los relevantes
  conocidos para esa consulta.
- **NDCG@10** da más peso a los aciertos de las primeras posiciones. Comparo
  el puntaje del ranking entregado con el mejor ranking posible según los qrels.

Primero calculo las métricas consulta por consulta y después saco el promedio.
Así cada consulta tiene el mismo peso en la comparación.

In [32]:
def EC_metricas(ranking, juicios, k=10):
    ganancias = [juicios.get(doc_id, 0) for doc_id in ranking[:k]]
    relevantes = sum(grado > 0 for grado in juicios.values())
    aciertos = sum(grado > 0 for grado in ganancias)

    def dcg(grados):
        return sum((2 ** grado - 1) / log2(posicion + 2)
                   for posicion, grado in enumerate(grados))

    ideal = dcg(sorted(juicios.values(), reverse=True)[:k])
    return {
        "precision": aciertos / k,
        "recall": aciertos / relevantes if relevantes else 0.0,
        "ndcg": dcg(ganancias) / ideal if ideal else 0.0,
    }


def EC_evaluar(buscador, consultas, qrels, modelo, k=10):
    por_consulta = {}
    for qid in sorted(qrels, key=int):
        puntajes = buscador.puntuar(consultas[qid], modelo, k)
        ranking = [buscador.ids[fila] for fila, score in puntajes]
        por_consulta[qid] = EC_metricas(ranking, qrels[qid], k)
    return por_consulta

EC_ids_test = sorted(EC_qrels, key=int)
print("IDs evaluados (test):")
for inicio in range(0, len(EC_ids_test), 20):
    print(", ".join(EC_ids_test[inicio:inicio + 20]))
print("\nPrimeras consultas de test:")
for qid in EC_ids_test[:3]:
    print(f"{qid}: {EC_consultas[qid]}")

EC_resultados = {
    modelo: EC_evaluar(EC_buscador, EC_consultas, EC_qrels, modelo)
    for modelo in ("tfidf", "bm25")
}
EC_nombres_metricas = ("precision", "recall", "ndcg")
EC_promedios = {
    modelo: {metrica: mean(fila[metrica] for fila in detalle.values())
             for metrica in EC_nombres_metricas}
    for modelo, detalle in EC_resultados.items()
}
print("\nModelo     Precision@10    Recall@10      NDCG@10")
for modelo, promedios in EC_promedios.items():
    valores = "   ".join(f"{promedios[metrica]:12.6f}" for metrica in EC_nombres_metricas)
    print(f"{modelo.upper():<8}{valores}")
print("\nConsultas por mejor NDCG@10:")
EC_comparacion = Counter()
for qid in EC_ids_test:
    tfidf = EC_resultados["tfidf"][qid]["ndcg"]
    bm25 = EC_resultados["bm25"][qid]["ndcg"]
    EC_comparacion["BM25" if bm25 > tfidf else "TF-IDF" if tfidf > bm25 else "Empate"] += 1
print(dict(EC_comparacion))
print("\nEjemplo de acierto en ambos modelos, consulta 3:",
      {m: EC_resultados[m]["3"] for m in EC_resultados})
print("Ejemplo sin aciertos en los primeros 10, consulta 13:",
      {m: EC_resultados[m]["13"] for m in EC_resultados})

IDs evaluados (test):
1, 3, 5, 13, 36, 42, 48, 49, 50, 51, 53, 54, 56, 57, 70, 72, 75, 94, 99, 100
113, 115, 118, 124, 127, 128, 129, 130, 132, 133, 137, 141, 142, 143, 146, 148, 163, 171, 179, 180
183, 185, 198, 208, 212, 213, 216, 217, 218, 219, 230, 232, 233, 236, 237, 238, 239, 248, 249, 261
268, 269, 274, 275, 279, 294, 295, 298, 300, 303, 312, 314, 324, 327, 338, 343, 350, 354, 362, 380
384, 385, 386, 388, 399, 410, 411, 415, 421, 431, 436, 437, 439, 440, 443, 452, 475, 478, 491, 501
502, 507, 508, 513, 514, 516, 517, 521, 525, 527, 528, 532, 533, 535, 536, 539, 540, 544, 549, 551
552, 554, 560, 569, 575, 577, 578, 587, 589, 593, 597, 598, 613, 619, 623, 628, 636, 637, 641, 644
649, 659, 660, 674, 684, 690, 691, 692, 693, 700, 702, 715, 716, 718, 721, 723, 727, 728, 729, 742
743, 744, 756, 759, 768, 770, 775, 781, 783, 784, 785, 793, 800, 805, 808, 811, 814, 820, 821, 823
830, 831, 832, 834, 837, 839, 845, 847, 852, 859, 870, 873, 879, 880, 882, 887, 903, 904, 907, 911
913, 914, 

| Modelo | Precision@10 | Recall@10 | NDCG@10 |
|---|---:|---:|---:|
| TF-IDF con coseno | 0,081667 | 0,734333 | 0,593846 |
| BM25 | 0,090333 | 0,818278 | 0,681405 |

En este corpus me quedo con **BM25** como modelo predeterminado. En promedio,
recuperó cerca del **81,83 %** de los relevantes de cada consulta en los
primeros 10 puestos. Con TF-IDF obtuve **73,43 %**, una diferencia de
**8,39 puntos porcentuales**. BM25 también mejoró el NDCG en **92 consultas**,
frente a **38** donde TF-IDF quedó arriba. En **170** tuvieron el mismo NDCG.

La precisión se ve baja incluso cuando el *recall* es alto. Esto tiene sentido
porque hay 339 juicios positivos para 300 consultas, cerca de un documento
relevante conocido por consulta. Si una consulta solo tiene un documento
relevante anotado y sale primero, su Precision@10 es 0,10 y su Recall@10 es
1,00. Eso pasa en la consulta 3 con BM25. Los dos modelos aciertan en esa
consulta, pero con BM25 el usuario lo ve antes.

En la consulta 13, ninguno encontró el documento anotado en los primeros 10.
El buscador usa coincidencias de palabras, así que dos textos pueden hablar
del mismo tema con términos diferentes y obtener una puntuación baja. Los
resultados muestran cómo rindieron estos modelos con SciFact y este
preprocesamiento.

## 5. Prueba del docente

**URL base** · [https://scifact-recuperacion-ec.pages.dev](https://scifact-recuperacion-ec.pages.dev)

El 18 de noviembre de 2026 probé [`GET /health`](https://scifact-recuperacion-ec.pages.dev/health) en la URL pública. Respondió con HTTP 200, estado `ok`, **5.183 documentos** y los modelos `bm25` y `tfidf`. También envié estas búsquedas a `POST /search`. Todas respondieron con HTTP 200 y los IDs coincidieron con los que obtuve en el notebook.

| Consulta de SciFact | Modelo | IDs en orden |
|---|---|---|
| 3 | BM25 | 14717500, 4414547, 2739854 |
| 3 | TF-IDF | 23389795, 2739854, 14717500 |
| 13 | BM25 | 7662395, 1263446, 17450673, 8842332, 12779444 |
| 13 | TF-IDF | 7662395, 17450673, 5487448, 16322674, 22922353 |

### Cómo probar la API en Postman

1. **Verificar que el servicio está activo.** Seleccione `GET`, pegue
   `https://scifact-recuperacion-ec.pages.dev/health` y presione **Send**.
   La respuesta esperada es `200 OK` con el estado `ok` y 5.183 documentos.

2. **Probar una búsqueda.** Seleccione `POST` y pegue la URL completa
   `https://scifact-recuperacion-ec.pages.dev/search`.
   En **Body → raw → JSON**, pegue lo siguiente y presione **Send**.

   ```json
   {
     "query": "1,000 genomes project enables mapping of genetic sequence variation consisting of rare variants with larger penetrance effects than common variants.",
     "model": "bm25",
     "top_k": 3
   }
   ```

   La respuesta esperada es `200 OK` con tres resultados. El primer
   `document_id` debe ser `14717500`.

3. **Comparar con TF-IDF.** Cambie `"model": "bm25"` por `"model": "tfidf"`
   y vuelva a enviar la solicitud. El primer `document_id` debe ser `23389795`.

In [33]:
EC_URL_PUBLICA = None


def EC_verificar_api(enviar):
    codigo, salud = enviar("/health")
    assert codigo == 200 and salud["status"] == "ok"
    assert salud["corpus"] == "SciFact" and salud["documents"] == len(EC_documentos)
    print("GET /health:", codigo, salud)
    # La segunda consulta no formó parte de la demostración de ranking de la sección 3.2.
    for qid, top_k in (("3", 10), ("13", 5)):
        for modelo in ("bm25", "tfidf"):
            solicitud = {"query": EC_consultas[qid], "model": modelo, "top_k": top_k}
            codigo, datos = enviar("/search", solicitud)
            esperados = EC_buscador.buscar(EC_consultas[qid], modelo, top_k)["results"]
            assert codigo == 200 and datos["model"] == modelo
            filas = datos["results"]
            ids = [fila["document_id"] for fila in filas]
            puntajes = [fila["score"] for fila in filas]
            assert len(filas) <= top_k and len(set(ids)) == len(ids)
            assert puntajes == sorted(puntajes, reverse=True)
            assert ids == [fila["document_id"] for fila in esperados]
            assert all(identificador in EC_documentos for identificador in ids)
            assert [fila["rank"] for fila in filas] == list(range(1, len(filas) + 1))
            assert all({"rank", "document_id", "score", "title", "snippet"} <= fila.keys()
                       for fila in filas)
            print(f"POST /search · {modelo} · consulta {qid}: {codigo}. "
                  f"Resultados: {len(ids)}. IDs: {', '.join(ids)}")
    codigo, error = enviar("/search", {
        "query": EC_consultas["13"], "model": "modelo-invalido", "top_k": 5,
    })
    assert codigo == 400 and "error" in error
    print("Modelo inválido:", codigo, error["error"])


def EC_enviar(ruta, solicitud=None):
    if EC_URL_PUBLICA:
        return EC_peticion(EC_URL_PUBLICA, ruta, solicitud)
    metodo = "GET" if solicitud is None else "POST"
    return EC_responder(EC_buscador, metodo, ruta, solicitud)


print("Verificación contra:", EC_URL_PUBLICA or "rutas del servicio en Python")
EC_verificar_api(EC_enviar)

Verificación contra: rutas del servicio en Python
GET /health: 200 {'status': 'ok', 'corpus': 'SciFact', 'documents': 5183, 'models': ['bm25', 'tfidf']}
POST /search · bm25 · consulta 3: 200. Resultados: 10. IDs: 14717500, 4414547, 2739854, 4378885, 19058822, 3672261, 4632921, 23389795, 2107238, 13519661
POST /search · tfidf · consulta 3: 200. Resultados: 10. IDs: 23389795, 2739854, 14717500, 4632921, 4378885, 3672261, 4414547, 19058822, 15153602, 12271486
POST /search · bm25 · consulta 13: 200. Resultados: 5. IDs: 7662395, 1263446, 17450673, 8842332, 12779444
POST /search · tfidf · consulta 13: 200. Resultados: 5. IDs: 7662395, 17450673, 5487448, 16322674, 22922353
Modelo inválido: 400 model debe ser bm25 o tfidf.


En las dos consultas, los IDs de la API coincidieron con los rankings
calculados en Python. También comprobé que los resultados respetan el tamaño
solicitado, tienen puntuaciones en orden descendente y no repiten documentos.
La respuesta fue 400 al solicitar un modelo inválido.

Para levantar la versión local y probarla desde Postman, se puede ejecutar
este bloque aparte. La URL queda disponible hasta que se interrumpa la celda.

~~~python
with EC_servidor_local(EC_buscador, puerto=8000) as servidor:
    print("http://127.0.0.1:8000")
    servidor.serve_forever()
~~~